# Production RAG capstone

**Track:** Enterprise Knowledge Assistant · **Stage:** Capstone

The capstone asks you to design a credible production RAG system for NovaTech. It is not one giant prompt. It is an offline knowledge pipeline plus an online answer pipeline, surrounded by evaluation, authorization, observability, and rollback controls.

## What you will build

- A deterministic implementation that runs without API keys.
- A visible trace of evidence, decisions, and failure modes.
- A production design note explaining how this maps to real RAG libraries and systems.

## Concept map

```mermaid
flowchart TD
  subgraph Offline["Offline knowledge pipeline"]
    S["Sources"] --> Parse["Parse + normalize"] --> Chunk["Chunk + metadata"] --> Index["Indexes: lexical / vector / graph"]
    Chunk --> EvalData["Golden set + fixtures"]
  end
  subgraph Online["Online answer pipeline"]
    Q["Question"] --> Auth["Access filter"] --> Route["Route"] --> Retrieve["Retrieve / rerank"] --> Generate["Answer with citations"] --> Verify["Verify or abstain"]
  end
  Index --> Retrieve
  EvalData --> Gate["Release gate"]
  Verify --> Trace["Trace: latency, cost, sources"]
```

## Setup

Run this notebook from the repository root, or open it in GitHub and copy cells into a local Jupyter session. The helper code lives in `src/enterprise_rag` so the notebook remains readable while the implementation stays testable.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

def show(obj):
    print(json.dumps(obj, indent=2))

The implementation below runs a small production-style trace. The numbers are toy estimates, but the shape is real: every request should report route, citations, support, latency, and cost.

In [ ]:
from src.enterprise_rag.lab_experiments import build_enterprise_chunks, production_run
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
questions = ["What increased by 14% in Q2 2025?", "What does AX-774-B mean?", "Who supplies Project Atlas technology and what regulation applies?", "What is NovaTech cafeteria menu today?"]
show(production_run(questions, chunks))

### Capstone checklist

Your final design should justify source inventory, ingestion strategy, chunking and metadata policy, retrieval routes, access control before retrieval, reranking strategy, citation verification, abstention policy, evaluation suite, trace schema, latency/cost budgets, freshness checks, rollback plan, and which optional libraries you would adopt now, later, or not at all.

## Deliberate failure case

Before moving on, make the system fail on purpose. Change one variable: chunk size, query wording, top-k, reranking terms, route choice, or evaluation labels. Write down whether the failure belongs to ingestion, retrieval, evidence selection, generation, authorization, or operations.

In [ ]:
# Try your own failure experiment here.
# Example: lower top_k to 1, ask an unsupported question, or remove an important query term.
from src.enterprise_rag.lab_experiments import build_enterprise_chunks
question = "What policy covers parental leave?"
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
print("Question:", question)
print("Now change the query, top_k, or chunking strategy and rerun a comparison helper.")

## Reflection questions

1. What did the simplest baseline get right?
2. What failure was invisible until you inspected the trace?
3. Which component would you improve first in production, and how would you prove it helped?
4. What should the system do when evidence is missing, unauthorized, stale, or contradictory?

## References and next reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Stanford IR book: https://nlp.stanford.edu/IR-book/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipeline docs: https://docs.haystack.deepset.ai/docs/pipelines
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/